In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### The Network Name in db and Ted's spread

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl

df["Station ID"] = pd.to_numeric(df["Station ID"], errors="coerce").astype("Int64")
df["Native ID"] = df["Native ID"].astype("string")

df_replace = df[df['ISSUE'].str.contains('replaced by', case=False, na=False)]
df_replace


df_replace =  df_replace[['ISSUE','Network ID', 'Station ID', 'Unique Names', 'Native ID']].reset_index(drop=True)

df_replace['new_native_id'] = df_replace['ISSUE'].str.extract(
    r'Replaced by\s+([A-Z0-9]+)',
    expand=False
)

df_replace = df_replace.rename(columns={
    'Native ID': 'old_native_id',
    'Station ID': 'old_station_id',
    'Unique Names': 'old_station_name'
})

df_replace

,ISSUE,Network ID,old_station_id,old_station_name,old_native_id,new_native_id
0,"Replaced by AGBP002 - move data record, delete...",10.0,1510,Bonaparte,bo107,AGBP002
1,"Replaced by AGCH005 - move data record, delete...",10.0,1511,Chase,ch107,AGCH005
2,"Replaced by AGCW006 - move data record, delete...",10.0,1512,Coldwater,co107,AGCW006
3,"Replaced by AGDC007 - move data record, delete...",10.0,1513,Deep Creek,de107,AGDC007
4,"Replaced by AGDL009 - move data record, delete...",10.0,1515,Douglas Lake,do107,AGDL009
5,"Replaced by AGDS008 - move data record, delete...",10.0,1514,Diamond S,di107,AGDS008
6,"Replaced by AGGR012 - move data record, delete...",10.0,1516,Grindrod,gr107,AGGR012
7,"Replaced by AGHC014 - move data record, delete...",10.0,1518,Hat Creek,hat107,AGHC014
8,"Replaced by AGHR013 - move data record, delete...",10.0,1517,Halfway Ranch,ha107,AGHR013
9,"Replaced by AGLN018 - move data record, delete...",10.0,1522,Sunshine Valley,su107,AGLN018


### check the obs records for both the old and new ID, then update the historical id in obs_raw, delete old station

In [5]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id = %s) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id = %s) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id = %s
     ) t) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN meta_station s_old ON m_old.station_id = s_old.station_id
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
     JOIN meta_station s_new ON m_new.station_id = s_new.station_id
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE s_old.native_id = %s
       AND s_new.native_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;

"""


def count_station_stats(old_id, new_id, engine):

    params = (
    old_id,          # n_old
    new_id,          # n_new
    old_id, new_id,  # n_overlap
    old_id, new_id   # n_overlap_same_datum
    )

    df = pd.read_sql(
        query,
        engine,
        params = params
    )
    return df.iloc[0]

# df_test = df_replace.head(3).copy()


stats = df_replace.apply(
    lambda r: count_station_stats(r['old_native_id'], r['new_native_id'], engine),
    axis=1
)

df_replace[['n_old', 'n_new', 'n_overlap', 'n_overlap_same_datum']] = stats
df_replace

,ISSUE,Network ID,old_station_id,old_station_name,old_native_id,new_native_id,n_old,n_new,n_overlap,n_overlap_same_datum
0,"Replaced by AGBP002 - move data record, delete...",10.0,1510,Bonaparte,bo107,AGBP002,0,0,0,0
1,"Replaced by AGCH005 - move data record, delete...",10.0,1511,Chase,ch107,AGCH005,0,0,0,0
2,"Replaced by AGCW006 - move data record, delete...",10.0,1512,Coldwater,co107,AGCW006,0,0,0,0
3,"Replaced by AGDC007 - move data record, delete...",10.0,1513,Deep Creek,de107,AGDC007,0,0,0,0
4,"Replaced by AGDL009 - move data record, delete...",10.0,1515,Douglas Lake,do107,AGDL009,0,0,0,0
5,"Replaced by AGDS008 - move data record, delete...",10.0,1514,Diamond S,di107,AGDS008,0,0,0,0
6,"Replaced by AGGR012 - move data record, delete...",10.0,1516,Grindrod,gr107,AGGR012,0,0,0,0
7,"Replaced by AGHC014 - move data record, delete...",10.0,1518,Hat Creek,hat107,AGHC014,0,0,0,0
8,"Replaced by AGHR013 - move data record, delete...",10.0,1517,Halfway Ranch,ha107,AGHR013,0,0,0,0
9,"Replaced by AGLN018 - move data record, delete...",10.0,1522,Sunshine Valley,su107,AGLN018,0,0,0,0


In [6]:
query = """
SELECT
    -- old station native_id (from DB)
    (SELECT s.native_id
     FROM meta_station s
     WHERE s.station_id = %s
       AND s.network_id = %s
    ) AS old_native_id_db,

    -- old count (by old station_id and network)
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     JOIN meta_station s ON m.station_id = s.station_id
     WHERE m.station_id = %s
       AND s.network_id = %s
    ) AS n_old,

    -- new count (by new native_id and network)
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id = %s
       AND s.network_id = %s
    ) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         JOIN meta_station s ON m.station_id = s.station_id
         WHERE m.station_id = %s
           AND s.network_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id = %s
           AND s.network_id = %s
     ) t) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id
     JOIN meta_station s_old ON m_old.station_id = s_old.station_id

     JOIN meta_history m_new
     JOIN meta_station s_new ON m_new.station_id = s_new.station_id
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id
       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE m_old.station_id = %s
       AND s_old.network_id = %s
       AND s_new.native_id = %s
       AND s_new.network_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;
"""

def count_station_stats(old_station_id, old_network_id, new_native_id, new_network_id, engine):

    params = (
        int(old_station_id), old_network_id,  # old_native_id_db
        int(old_station_id), old_network_id,  # n_old
        str(new_native_id), new_network_id,   # n_new
        int(old_station_id), old_network_id,  # n_overlap (old)
        str(new_native_id), new_network_id,   # n_overlap (new)
        int(old_station_id), old_network_id,  # n_overlap_same_datum (old)
        str(new_native_id), new_network_id    # n_overlap_same_datum (new)
    )

    df = pd.read_sql(query, engine, params=params)
    return df.iloc[0]

stats = df_replace.apply(
    lambda r: count_station_stats(
        r['old_station_id'],
        r['Network ID'],
        r['new_native_id'],
        r['Network ID'],
        engine
    ),
    axis=1
)

df_replace[
    [
        'old_native_id_db',
        'n_old',
        'n_new',
        'n_overlap',
        'n_overlap_same_datum'
    ]
] = stats
df_replace

,ISSUE,Network ID,old_station_id,old_station_name,old_native_id,new_native_id,n_old,n_new,n_overlap,n_overlap_same_datum,old_native_id_db
0,"Replaced by AGBP002 - move data record, delete...",10.0,1510,Bonaparte,bo107,AGBP002,18177,0,0,0,BP002
1,"Replaced by AGCH005 - move data record, delete...",10.0,1511,Chase,ch107,AGCH005,17701,0,0,0,CHASESTN
2,"Replaced by AGCW006 - move data record, delete...",10.0,1512,Coldwater,co107,AGCW006,16423,0,0,0,COLDWATR
3,"Replaced by AGDC007 - move data record, delete...",10.0,1513,Deep Creek,de107,AGDC007,18208,0,0,0,DEEPCREE
4,"Replaced by AGDL009 - move data record, delete...",10.0,1515,Douglas Lake,do107,AGDL009,22112,0,0,0,DOUGLAKE
5,"Replaced by AGDS008 - move data record, delete...",10.0,1514,Diamond S,di107,AGDS008,11574,0,0,0,DIAMONDS
6,"Replaced by AGGR012 - move data record, delete...",10.0,1516,Grindrod,gr107,AGGR012,18724,0,0,0,GRINROD
7,"Replaced by AGHC014 - move data record, delete...",10.0,1518,Hat Creek,hat107,AGHC014,14447,0,0,0,HATCREEK
8,"Replaced by AGHR013 - move data record, delete...",10.0,1517,Halfway Ranch,ha107,AGHR013,16228,0,0,0,HALFRNCH
9,"Replaced by AGLN018 - move data record, delete...",10.0,1522,Sunshine Valley,su107,AGLN018,21069,0,0,0,LOWNICOL


### Process for df_replace

### First check if all old stations exist

In [9]:
from sqlalchemy import text

exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_station
    WHERE native_id = :native_id
)
""")

exists_list = []

with engine.connect() as conn:   # reuse ONE connection
    for i, row in df_replace.iterrows():
        native_id = row['old_native_id']

        exists = conn.execute(
            exists_sql,
            {"native_id": native_id}
        ).scalar()

        status = "EXISTS" if exists else "NOT EXISTS"
        exists_list.append(status)

        print(
            f"[{i+1:>3}/{len(df_replace)}] "
            f"old_native_id={native_id:<15} | "
            f"→ {status}"
        )

# add column to DataFrame
df_replace['old_station_status'] = exists_list

[  1/22] old_native_id=bo107           | → NOT EXISTS
[  2/22] old_native_id=ch107           | → NOT EXISTS
[  3/22] old_native_id=co107           | → NOT EXISTS
[  4/22] old_native_id=de107           | → NOT EXISTS
[  5/22] old_native_id=do107           | → NOT EXISTS
[  6/22] old_native_id=di107           | → NOT EXISTS
[  7/22] old_native_id=gr107           | → NOT EXISTS
[  8/22] old_native_id=hat107          | → NOT EXISTS
[  9/22] old_native_id=ha107           | → NOT EXISTS
[ 10/22] old_native_id=su107           | → NOT EXISTS
[ 11/22] old_native_id=ma107           | → NOT EXISTS
[ 12/22] old_native_id=SBC25           | → NOT EXISTS
[ 13/22] old_native_id=25672           | → NOT EXISTS
[ 14/22] old_native_id=SBC15           | → NOT EXISTS
[ 15/22] old_native_id=24547           | → NOT EXISTS
[ 16/22] old_native_id=SBC19           | → NOT EXISTS
[ 17/22] old_native_id=qu107           | → NOT EXISTS
[ 18/22] old_native_id=si107           | → NOT EXISTS
[ 19/22] old_native_id=so107

In [10]:
from sqlalchemy import text

exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_station
    WHERE station_id = :station_id
)
""")

exists_list = []

with engine.connect() as conn:   # reuse ONE connection
    for i, row in df_replace.iterrows():
        station_id = row['old_station_id']

        exists = conn.execute(
            exists_sql,
            {"station_id": station_id}
        ).scalar()

        status = "EXISTS" if exists else "NOT EXISTS"
        exists_list.append(status)

        print(
            f"[{i+1:>3}/{len(df_replace)}] "
            f"old_station_id={station_id:<15} | "
            f"→ {status}"
        )

# add column to DataFrame
df_replace['old_station_status'] = exists_list

[  1/22] old_station_id=1510            | → EXISTS
[  2/22] old_station_id=1511            | → EXISTS
[  3/22] old_station_id=1512            | → EXISTS
[  4/22] old_station_id=1513            | → EXISTS
[  5/22] old_station_id=1515            | → EXISTS
[  6/22] old_station_id=1514            | → EXISTS
[  7/22] old_station_id=1516            | → EXISTS
[  8/22] old_station_id=1518            | → EXISTS
[  9/22] old_station_id=1517            | → EXISTS
[ 10/22] old_station_id=1522            | → EXISTS
[ 11/22] old_station_id=3284            | → EXISTS
[ 12/22] old_station_id=3107            | → EXISTS
[ 13/22] old_station_id=1526            | → EXISTS
[ 14/22] old_station_id=3101            | → EXISTS
[ 15/22] old_station_id=1523            | → NOT EXISTS
[ 16/22] old_station_id=3102            | → NOT EXISTS
[ 17/22] old_station_id=1519            | → EXISTS
[ 18/22] old_station_id=1509            | → EXISTS
[ 19/22] old_station_id=1520            | → EXISTS
[ 20/22] old_station_id

### Check if the new station exist

In [11]:
from sqlalchemy import text

exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_history m
    JOIN meta_station s ON m.station_id = s.station_id
    WHERE s.native_id = :native_id
    AND s.network_id = :network_id
)
""")

exists_list = []

with engine.connect() as conn:   # reuse ONE connection
    for i, row in df_replace.iterrows():
        native_id = row['new_native_id']
        network_id = row['Network ID']

        exists = conn.execute(
            exists_sql,
            {"native_id": native_id,
            "network_id": network_id}
        ).scalar()

        status = "EXISTS" if exists else "NOT EXISTS"
        exists_list.append(status)

        print(
            f"[{i+1:>3}/{len(df_replace)}] "
            f"new_native_id={native_id:<15} | "
            f"→ {status}"
        )

# add column to DataFrame
df_replace['new_station_status'] = exists_list

[  1/22] new_native_id=AGBP002         | → EXISTS
[  2/22] new_native_id=AGCH005         | → EXISTS
[  3/22] new_native_id=AGCW006         | → EXISTS
[  4/22] new_native_id=AGDC007         | → EXISTS
[  5/22] new_native_id=AGDL009         | → EXISTS
[  6/22] new_native_id=AGDS008         | → EXISTS
[  7/22] new_native_id=AGGR012         | → EXISTS
[  8/22] new_native_id=AGHC014         | → EXISTS
[  9/22] new_native_id=AGHR013         | → EXISTS
[ 10/22] new_native_id=AGLN018         | → EXISTS
[ 11/22] new_native_id=AGML020         | → EXISTS
[ 12/22] new_native_id=AGOKNGKMOS      | → EXISTS
[ 13/22] new_native_id=AGOKNGOLVS      | → EXISTS
[ 14/22] new_native_id=AGOKNGOYMA      | → EXISTS
[ 15/22] new_native_id=AGOKNGPNRA      | → EXISTS
[ 16/22] new_native_id=AGOKNGVNBX      | → EXISTS
[ 17/22] new_native_id=AGQC023         | → EXISTS
[ 18/22] new_native_id=AGSH024         | → EXISTS
[ 19/22] new_native_id=AGST025         | → EXISTS
[ 20/22] new_native_id=AGML019         | → EXISTS


In [12]:
df_replace

,ISSUE,Network ID,old_station_id,old_station_name,old_native_id,new_native_id,n_old,n_new,n_overlap,n_overlap_same_datum,old_native_id_db,old_station_status,new_station_status
0,"Replaced by AGBP002 - move data record, delete...",10.0,1510,Bonaparte,bo107,AGBP002,18177,0,0,0,BP002,EXISTS,EXISTS
1,"Replaced by AGCH005 - move data record, delete...",10.0,1511,Chase,ch107,AGCH005,17701,0,0,0,CHASESTN,EXISTS,EXISTS
2,"Replaced by AGCW006 - move data record, delete...",10.0,1512,Coldwater,co107,AGCW006,16423,0,0,0,COLDWATR,EXISTS,EXISTS
3,"Replaced by AGDC007 - move data record, delete...",10.0,1513,Deep Creek,de107,AGDC007,18208,0,0,0,DEEPCREE,EXISTS,EXISTS
4,"Replaced by AGDL009 - move data record, delete...",10.0,1515,Douglas Lake,do107,AGDL009,22112,0,0,0,DOUGLAKE,EXISTS,EXISTS
5,"Replaced by AGDS008 - move data record, delete...",10.0,1514,Diamond S,di107,AGDS008,11574,0,0,0,DIAMONDS,EXISTS,EXISTS
6,"Replaced by AGGR012 - move data record, delete...",10.0,1516,Grindrod,gr107,AGGR012,18724,0,0,0,GRINROD,EXISTS,EXISTS
7,"Replaced by AGHC014 - move data record, delete...",10.0,1518,Hat Creek,hat107,AGHC014,14447,0,0,0,HATCREEK,EXISTS,EXISTS
8,"Replaced by AGHR013 - move data record, delete...",10.0,1517,Halfway Ranch,ha107,AGHR013,16228,0,0,0,HALFRNCH,EXISTS,EXISTS
9,"Replaced by AGLN018 - move data record, delete...",10.0,1522,Sunshine Valley,su107,AGLN018,21069,0,0,0,LOWNICOL,EXISTS,EXISTS


#### For the pairs old station not exist, new station exist, seems means has been replaced, so don't need further process

#### For the pairs that old station exist, new station exist, change the old historical id in obs_raw to the new station

In [13]:
df_replace_exists = df_replace[
    (df_replace['old_station_status'] == "EXISTS") & 
    (df_replace['new_station_status'] == "EXISTS")
]

df_replace_exists

,ISSUE,Network ID,old_station_id,old_station_name,old_native_id,new_native_id,n_old,n_new,n_overlap,n_overlap_same_datum,old_native_id_db,old_station_status,new_station_status
0,"Replaced by AGBP002 - move data record, delete...",10.0,1510,Bonaparte,bo107,AGBP002,18177,0,0,0,BP002,EXISTS,EXISTS
1,"Replaced by AGCH005 - move data record, delete...",10.0,1511,Chase,ch107,AGCH005,17701,0,0,0,CHASESTN,EXISTS,EXISTS
2,"Replaced by AGCW006 - move data record, delete...",10.0,1512,Coldwater,co107,AGCW006,16423,0,0,0,COLDWATR,EXISTS,EXISTS
3,"Replaced by AGDC007 - move data record, delete...",10.0,1513,Deep Creek,de107,AGDC007,18208,0,0,0,DEEPCREE,EXISTS,EXISTS
4,"Replaced by AGDL009 - move data record, delete...",10.0,1515,Douglas Lake,do107,AGDL009,22112,0,0,0,DOUGLAKE,EXISTS,EXISTS
5,"Replaced by AGDS008 - move data record, delete...",10.0,1514,Diamond S,di107,AGDS008,11574,0,0,0,DIAMONDS,EXISTS,EXISTS
6,"Replaced by AGGR012 - move data record, delete...",10.0,1516,Grindrod,gr107,AGGR012,18724,0,0,0,GRINROD,EXISTS,EXISTS
7,"Replaced by AGHC014 - move data record, delete...",10.0,1518,Hat Creek,hat107,AGHC014,14447,0,0,0,HATCREEK,EXISTS,EXISTS
8,"Replaced by AGHR013 - move data record, delete...",10.0,1517,Halfway Ranch,ha107,AGHR013,16228,0,0,0,HALFRNCH,EXISTS,EXISTS
9,"Replaced by AGLN018 - move data record, delete...",10.0,1522,Sunshine Valley,su107,AGLN018,21069,0,0,0,LOWNICOL,EXISTS,EXISTS


In [14]:
from sqlalchemy import text

SQL_DELETE_DUPLICATES = text("""
DELETE FROM obs_raw o
USING obs_raw o2
WHERE o.vars_id = o2.vars_id
  AND o.obs_time = o2.obs_time
  AND o.history_id = :old_hist_id
  AND o2.history_id = :new_hist_id
""")

SQL_BULK_MOVE = text("""
UPDATE obs_raw
SET history_id = :new_hist_id
WHERE history_id = :old_hist_id
""")

SQL_GET_HISTORY_BY_STATION_ID = text("""
SELECT h.history_id
FROM meta_history h
WHERE h.station_id = :station_id
ORDER BY h.history_id DESC
LIMIT 1
""")

SQL_GET_HISTORY_BY_NATIVE = text("""
SELECT h.history_id
FROM meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE s.native_id  = :native_id
  AND s.network_id = :network_id
ORDER BY h.history_id DESC
LIMIT 1
""")

def move_station_data_fast(old_station_id, new_native_id, new_network_id, engine):
    with engine.begin() as conn:

        # 🔹 old station: constrain by station_id
        old_hist_id = conn.execute(
            SQL_GET_HISTORY_BY_STATION_ID,
            {"station_id": old_station_id}
        ).scalar()

        # 🔹 new station: constrain by native_id + network_id
        new_hist_id = conn.execute(
            SQL_GET_HISTORY_BY_NATIVE,
            {
                "native_id": new_native_id,
                "network_id": new_network_id,
            }
        ).scalar()

        if old_hist_id is None or new_hist_id is None:
            print(f"❌ Missing history: {old_station_id} → {new_native_id}")
            return 0

        # 1️⃣ delete duplicates
        dup_res = conn.execute(
            SQL_DELETE_DUPLICATES,
            {
                "old_hist_id": old_hist_id,
                "new_hist_id": new_hist_id,
            }
        )

        # 2️⃣ bulk reassign
        move_res = conn.execute(
            SQL_BULK_MOVE,
            {
                "old_hist_id": old_hist_id,
                "new_hist_id": new_hist_id,
            }
        )

        print(
            f"    🧹 removed duplicates={dup_res.rowcount}, "
            f"🚚 moved={move_res.rowcount}"
        )

        return move_res.rowcount

results = []

for i, row in df_replace_exists.iterrows():
    old_station_id = row["old_station_id"]
    new_native_id  = row["new_native_id"]
    new_network_id = row["Network ID"]

    print(f"[{i+1}/{len(df_replace_exists)}] Moving station_id {old_station_id} → native_id {new_native_id}")

    n_moved = move_station_data_fast(
        old_station_id,
        new_native_id,
        new_network_id,
        engine
    )
    results.append(n_moved)

print("All done!")


[1/19] Moving station_id 1510 → native_id AGBP002
    🧹 removed duplicates=0, 🚚 moved=18177
[2/19] Moving station_id 1511 → native_id AGCH005
    🧹 removed duplicates=0, 🚚 moved=17701
[3/19] Moving station_id 1512 → native_id AGCW006
    🧹 removed duplicates=0, 🚚 moved=16423
[4/19] Moving station_id 1513 → native_id AGDC007
    🧹 removed duplicates=0, 🚚 moved=18208
[5/19] Moving station_id 1515 → native_id AGDL009
    🧹 removed duplicates=0, 🚚 moved=22112
[6/19] Moving station_id 1514 → native_id AGDS008
    🧹 removed duplicates=0, 🚚 moved=11574
[7/19] Moving station_id 1516 → native_id AGGR012
    🧹 removed duplicates=0, 🚚 moved=18724
[8/19] Moving station_id 1518 → native_id AGHC014
    🧹 removed duplicates=0, 🚚 moved=14447
[9/19] Moving station_id 1517 → native_id AGHR013
    🧹 removed duplicates=0, 🚚 moved=16228
[10/19] Moving station_id 1522 → native_id AGLN018
    🧹 removed duplicates=0, 🚚 moved=21069
[11/19] Moving station_id 3284 → native_id AGML020
    🧹 removed duplicates=0, 

### Delete the old stations

In [15]:
from tqdm import tqdm
import sqlalchemy as sa

station_ids = (
    df_replace_exists["old_station_id"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
# station_ids[0]


# List of station_ids to delete
station_ids_to_delete = station_ids  # or a subset

delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for sid in tqdm(station_ids_to_delete, desc="Deleting stations"):

        # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
        res_flags = conn.execute(
            delete_obs_flags,
            {"station_id": sid}
        )

        # 2️⃣ obs_raw
        res_obs = conn.execute(
            delete_obs,
            {"station_id": sid}
        )

        # 3️⃣ obs_derived_values (depends on meta_history)
        res_obs_dv = conn.execute(
            delete_obs_derived,
            {"station_id": sid}
        )

        # 4️⃣ meta_history
        res_hist = conn.execute(
            delete_history,
            {"station_id": sid}
        )

        # 5️⃣ meta_station
        res_sta = conn.execute(
            delete_station,
            {"station_id": sid}
        )

        tqdm.write(
            f"Station {sid}: "
            f"flags={res_flags.rowcount}, "
            f"obs_raw={res_obs.rowcount}, "
            f"obs_derived={res_obs_dv.rowcount}, "
            f"meta_history={res_hist.rowcount}, "
            f"meta_station={res_sta.rowcount}"
        )

Deleting stations:  21%|██        | 4/19 [00:00<00:00, 15.15it/s]

Station 1510: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 1511: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 1512: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 1513: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1


Deleting stations:  42%|████▏     | 8/19 [00:00<00:00, 16.16it/s]

Station 1515: flags=0, obs_raw=0, obs_derived=22, meta_history=1, meta_station=1
Station 1514: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 1516: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 1518: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1


Deleting stations:  68%|██████▊   | 13/19 [00:00<00:00, 18.57it/s]

Station 1517: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 1522: flags=0, obs_raw=0, obs_derived=11, meta_history=1, meta_station=1
Station 3284: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 3107: flags=0, obs_raw=0, obs_derived=24, meta_history=1, meta_station=1
Station 1526: flags=0, obs_raw=0, obs_derived=11, meta_history=1, meta_station=1


Deleting stations:  84%|████████▍ | 16/19 [00:00<00:00, 21.60it/s]

Station 3101: flags=0, obs_raw=0, obs_derived=24, meta_history=1, meta_station=1
Station 1519: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 1509: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 1520: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 1508: flags=0, obs_raw=0, obs_derived=21, meta_history=1, meta_station=1


Deleting stations: 100%|██████████| 19/19 [00:01<00:00, 16.86it/s]


Station 2452: flags=0, obs_raw=0, obs_derived=36, meta_history=1, meta_station=1
